# Stage 2.1 — LIBERO demonstration preparation
Run cells in order on a fresh Colab Enterprise Linux runtime (Python 3.10–3.12). Extraction is CPU-only; the requested V100 check is a separate preflight. No model weights, simulator installation, embeddings or retrieval are needed.

**Cloud execution: UNVERIFIED.** Local real-data validation is documented in `docs/STAGE2_1_HANDOFF.md`. Native LIBERO observations are post-action; output pairs `obs[i]` with `action[i+1]`. Success is assumed from expert provenance, never inferred from terminal rewards.

Before running, publish the local `stage2-memory` commit or upload the repository ZIP. Configure `REPO_DIR`, `SOURCE_REF`, and optional `GCS_OUTPUT_URI` below. Use an immutable published commit as `SOURCE_REF` for reproducibility. GCS uses runtime Application Default Credentials; grant the runtime identity access to your chosen bucket outside this notebook. No credentials belong in code.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, json, platform
REPO_URL = "https://github.com/wcysunrise/cs6758-vla-project.git"
SOURCE_REF = "stage2-memory"  # Prefer the published commit SHA from the handoff.
REPO_DIR = Path.home() / "cs6758-vla-project"
# If you uploaded/unzipped the local checkout, point REPO_DIR to it; it will not be modified.
GCS_OUTPUT_URI = ""  # e.g. gs://YOUR_BUCKET/cs6758/stage2_1 (no credentials)
LOCAL_OUTPUT_ROOT = Path.home() / "stage2_outputs"
REQUIRE_GPU = True  # Set False for CPU-only data preparation.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_ID
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "checkout", SOURCE_REF], cwd=REPO_DIR, check=True)
assert (REPO_DIR / "memory/extract_demonstrations.py").exists(), "Publish stage2-memory or upload the completed repository first"
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Repository:", REPO_DIR, "Output:", RUN_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True) if (REPO_DIR / ".git").exists() else "Uploaded archive")

In [ ]:
assert platform.system() == "Linux", "Cloud notebook targets Linux"
assert (3, 10) <= sys.version_info[:2] <= (3, 12), "Use Python 3.10–3.12 for the pinned torch build"
print(platform.platform(), sys.version)
import shutil
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
VENV = REPO_DIR / ".venv-stage2"
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
PYTHON = str(VENV / "bin/python")
subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-stage2.txt")], check=True)
# Same torch CUDA build used successfully by Stage 1 on V100; inspect and test below.
subprocess.run([PYTHON, "-m", "pip", "install", "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
if GCS_OUTPUT_URI:
    subprocess.run([PYTHON, "-m", "pip", "install", "google-cloud-storage==3.4.0"], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
(RUN_DIR / "installed_versions.txt").write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))

In [ ]:
preflight = r"""
import json, platform, sys, torch
from pathlib import Path
info = {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
        'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available(),
        'compiled_architectures': torch.cuda.get_arch_list()}
assert torch.__version__.split('+')[0] == '2.7.1'
if torch.cuda.is_available():
    info['gpu'] = torch.cuda.get_device_name(0)
    info['capability'] = list(torch.cuda.get_device_capability(0))
    if info['capability'] == [7, 0]:
        assert 'sm_70' in info['compiled_architectures'], 'Installed wheel lacks native V100 support'
    x = torch.ones((64, 64), device='cuda', dtype=torch.float32)
    y = x @ x
    torch.cuda.synchronize()
    assert torch.all(y == 64).item()
    info['cuda_matmul'] = 'PASS'
else:
    info['cuda_matmul'] = 'UNVERIFIED: no GPU'
Path(sys.argv[1]).write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
if sys.argv[2] == 'True':
    assert torch.cuda.is_available(), 'GPU required; attach V100 or explicitly choose CPU mode'
"""
subprocess.run([PYTHON, "-c", preflight, str(RUN_DIR / "hardware.json"), str(REQUIRE_GPU)], check=True)

In [ ]:
# All tests, including unchanged Stage 1 protocol tests.
result = subprocess.run([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO_DIR, text=True, capture_output=True)
(RUN_DIR / "tests.log").write_text(result.stdout + result.stderr)
print(result.stdout + result.stderr)
result.check_returncode()
# Download only the pinned task; the raw file remains unchanged and reusable.
OUTPUT = RUN_DIR / "data"
with (RUN_DIR / "extraction.log").open("w") as log:
    subprocess.run([PYTHON, "-m", "memory.extract_demonstrations", "--download",
                    "--raw-dir", str(LOCAL_OUTPUT_ROOT / "raw"), "--output", str(OUTPUT),
                    "--seed", "6758", "--eval-fraction", "0.2"], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, check=True)
print((OUTPUT / "dataset_metadata.json").read_text())

In [ ]:
validation = "from memory.dataset_utils import validate_outputs; import sys,json; print(json.dumps(validate_outputs(sys.argv[1]),indent=2))"
result = subprocess.check_output([PYTHON, "-c", validation, str(OUTPUT)], cwd=REPO_DIR, text=True)
(RUN_DIR / "validation.json").write_text(result)
print(result)
# Capture exact source code for archive-based runs as well as Git provenance.
import hashlib
source_files = [*sorted((REPO_DIR / "memory").glob("*.py")), REPO_DIR / "requirements-stage2.txt", REPO_DIR / "notebooks/stage2_1_prepare_data.ipynb"]
(RUN_DIR / "source_sha256.json").write_text(json.dumps({str(p.relative_to(REPO_DIR)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}, indent=2))

In [ ]:
# Export every generated output and log. The original dataset can be redownloaded
# using the pinned revision and hash in metadata; it is retained in LOCAL_OUTPUT_ROOT/raw.
archive = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("Archive:", archive, "bytes:", archive.stat().st_size)
if GCS_OUTPUT_URI:
    upload = r"""
from google.cloud import storage
from urllib.parse import urlparse
from pathlib import Path
import sys
uri = urlparse(sys.argv[1])
assert uri.scheme == 'gs' and uri.netloc, 'Expected gs://bucket/prefix'
path = Path(sys.argv[2])
blob = storage.Client().bucket(uri.netloc).blob(uri.path.strip('/') + '/' + path.name)
blob.upload_from_filename(str(path), if_generation_match=0, checksum='auto')
blob.reload()
assert blob.size == path.stat().st_size
print('Persisted gs://' + uri.netloc + '/' + blob.name, 'generation', blob.generation)
"""
    subprocess.run([PYTHON, "-c", upload, GCS_OUTPUT_URI, str(archive)], check=True)
else:
    print("No cloud persistence configured. Download this archive before stopping the runtime.")
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        from IPython.display import FileLink, display
        display(FileLink(str(archive)))
        print("If the link is inaccessible, download the printed archive path through the notebook file browser.")

## Completion check
Confirm the archive is in your bucket or on your computer before stopping the runtime. A download request is not proof that a download finished. Retain `dataset_metadata.json`, `split_manifest.json`, both HDF5 files and logs together. Review quarantined counts (default: malformed records fail extraction).

Stage 2.2 must preserve trajectory boundaries, fit any new statistics only on training trajectories, handle native image orientation/resolution explicitly, and compare actions in a shared coordinate/normalization convention. Dataset success and simulator replay remain unverified; this notebook performs no robot rollouts.